# 00 — Initial exploration of the Living Planet Database

This notebook is the starting point of the biodiversity analysis project.  
It introduces the Living Planet Database 2024 and checks whether the dataset is suitable for answering the main research question:

**Which vertebrate groups and world regions show the strongest population declines?**

At this stage, the goal is **not** to calculate trends or classify risk yet.  
Instead, this notebook focuses on understanding:

1. the structure of the dataset;
2. the main unit of analysis;
3. taxonomic, regional, ecosystem, and temporal coverage;
4. potential data-quality limitations that should be considered before trend calculation.

The following notebooks build on this exploration:

- `01_population_trend_calculation.ipynb` calculates population-level trends;
- `02_risk_analysis.ipynb` compares risk patterns across classes, ecosystems, and species summaries;
- `03_spatial_analysis.ipynb` examines where high-risk populations are geographically concentrated.


## 1. Load libraries and data

The analysis uses `pandas` for tabular data handling and `matplotlib` for simple exploratory visualizations.  
The raw Living Planet Database file is loaded from the `data/raw` folder.


In [ ]:
# Import the libraries used in this notebook.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Show all columns when displaying pandas DataFrames.
pd.set_option("display.max_columns", None)

# Define the path to the raw Living Planet Database CSV file.
DATA_PATH = Path("../data/raw/LivingPlanetIndex_2024_PublicData/LPD_2024_public.csv")

# Load the raw dataset into a pandas DataFrame.
lpd = pd.read_csv(DATA_PATH)

# Display the first rows to confirm that the file loaded correctly.
lpd.head()


## 2. Basic dataset structure

First, we check the overall shape of the dataset, the available columns, and their data types.  
This helps clarify what kind of information is available before any calculations are performed.


In [ ]:
# Check the number of rows and columns in the dataset.
# Each row represents a monitored population record.
lpd.shape


In [ ]:
# Display the column names to understand what variables are available.
lpd.columns.tolist()


In [ ]:
# Inspect data types and non-null counts for all columns.
# This helps identify missing values and potential type issues.
lpd.info()


## 3. Unit of analysis

The Living Planet Database is structured around **monitored populations**, not only species.  
This is important for the whole project: the main analysis should therefore be population-level.

Here we compare the number of:

- population records (`ID`),
- species names,
- binomial names.


In [ ]:
# Count unique population IDs, species labels, and binomial names.
# This clarifies the main unit of analysis and the relationship between populations and species.
unit_summary = pd.DataFrame({
    "Metric": [
        "Unique population records (ID)",
        "Unique species names",
        "Unique binomial names"
    ],
    "Count": [
        lpd["ID"].nunique(),
        lpd["Species"].nunique(),
        lpd["Binomial"].nunique()
    ]
})

unit_summary


**Interpretation:**  
The number of population records is larger than the number of species because one species can be monitored in multiple locations or studies.  
For this reason, later trend and risk calculations use **population records** as the main analytical unit.


## 4. Taxonomic coverage

Next, we examine how many population records are available for each vertebrate class.  
This helps determine whether comparisons across classes are meaningful and whether some groups are underrepresented.


In [ ]:
# Count the number of monitored population records per vertebrate class.
class_counts = (
    lpd["Class"]
    .value_counts(dropna=False)
    .rename_axis("Class")
    .reset_index(name="Population_Records")
)

class_counts


In [ ]:
# Visualize taxonomic coverage by class.
# This is an exploratory count plot, not a trend or risk result.
class_counts_sorted = class_counts.sort_values("Population_Records", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(class_counts_sorted["Class"].astype(str), class_counts_sorted["Population_Records"])
plt.xlabel("Number of population records")
plt.ylabel("Vertebrate class")
plt.title("Taxonomic coverage in the Living Planet Database")
plt.tight_layout()
plt.show()


## 5. Regional and ecosystem coverage

The main research question includes world regions, so it is important to check how population records are distributed geographically and across ecosystems.

This section looks at:

- broad regions;
- IPBES regions;
- ecosystem systems.


In [ ]:
# Count population records by broad region.
region_counts = (
    lpd["Region"]
    .value_counts(dropna=False)
    .rename_axis("Region")
    .reset_index(name="Population_Records")
)

region_counts


In [ ]:
# Visualize broad regional coverage.
# This shows data availability, not population decline.
region_counts_sorted = region_counts.sort_values("Population_Records", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(region_counts_sorted["Region"].astype(str), region_counts_sorted["Population_Records"])
plt.xlabel("Number of population records")
plt.ylabel("Region")
plt.title("Regional coverage in the Living Planet Database")
plt.tight_layout()
plt.show()


In [ ]:
# Count population records by IPBES region.
# IPBES regions provide an additional geographic grouping used in biodiversity reporting.
ipbes_counts = (
    lpd["IPBES_region"]
    .value_counts(dropna=False)
    .rename_axis("IPBES_region")
    .reset_index(name="Population_Records")
)

ipbes_counts


In [ ]:
# Count population records by ecosystem system.
# This supports later comparisons between terrestrial, freshwater, and marine populations.
system_counts = (
    lpd["System"]
    .value_counts(dropna=False)
    .rename_axis("System")
    .reset_index(name="Population_Records")
)

system_counts


In [ ]:
# Visualize ecosystem coverage.
# This helps show whether some systems have much more monitoring data than others.
system_counts_sorted = system_counts.sort_values("Population_Records", ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(system_counts_sorted["System"].astype(str), system_counts_sorted["Population_Records"])
plt.xlabel("Number of population records")
plt.ylabel("System")
plt.title("Ecosystem coverage in the Living Planet Database")
plt.tight_layout()
plt.show()


## 6. Temporal coverage

Trend analysis requires population observations across years.  
The Living Planet Database stores annual abundance values in year columns.  
Here we identify those year columns and check the temporal coverage of the dataset.


In [ ]:
# Identify columns that represent years.
# In this dataset, annual abundance columns are stored as column names such as "1970", "1971", etc.
year_columns = [
    col for col in lpd.columns
    if str(col).isdigit() and 1900 <= int(col) <= 2100
]

# Convert year column names to integers for easier summaries.
years = [int(col) for col in year_columns]

# Print a short summary of temporal coverage.
print("Number of year columns:", len(year_columns))
print("First year:", min(years))
print("Last year:", max(years))


In [ ]:
# Count how many non-missing abundance values exist for each year.
# This shows how monitoring coverage changes over time.
yearly_observation_counts = (
    lpd[year_columns]
    .notna()
    .sum()
    .rename_axis("Year")
    .reset_index(name="Observed_Populations")
)

yearly_observation_counts["Year"] = yearly_observation_counts["Year"].astype(int)

yearly_observation_counts.head()


In [ ]:
# Plot the number of observed population values by year.
# This is important because trend calculations are more reliable when temporal coverage is sufficient.
plt.figure(figsize=(10, 5))
plt.plot(
    yearly_observation_counts["Year"],
    yearly_observation_counts["Observed_Populations"]
)
plt.xlabel("Year")
plt.ylabel("Number of observed population values")
plt.title("Temporal coverage of population observations")
plt.tight_layout()
plt.show()


## 7. Missing values in key columns

Before calculating trends, we check missing values in variables that are important for later analysis:

- population ID;
- species and binomial names;
- class;
- system;
- region;
- coordinates.

This helps identify possible limitations for taxonomic and spatial analysis.


In [ ]:
# Select key columns used across the project.
# Only columns that exist in the dataset are included, making this check robust to minor schema differences.
key_columns = [
    "ID",
    "Species",
    "Binomial",
    "Class",
    "System",
    "Region",
    "IPBES_region",
    "Latitude",
    "Longitude"
]

existing_key_columns = [col for col in key_columns if col in lpd.columns]

# Calculate missing values and missing percentages for key columns.
missing_summary = pd.DataFrame({
    "Column": existing_key_columns,
    "Missing_Values": [lpd[col].isna().sum() for col in existing_key_columns],
    "Missing_Percent": [lpd[col].isna().mean() * 100 for col in existing_key_columns]
})

missing_summary.sort_values("Missing_Percent", ascending=False)


## 8. Coverage across class and region

The main project question connects taxonomic groups and world regions.  
A cross-tabulation helps show whether all classes are represented across all regions or whether some combinations have limited data.


In [ ]:
# Create a cross-tabulation of population records by class and region.
# This table shows data coverage for class-region combinations.
class_region_counts = pd.crosstab(
    lpd["Class"],
    lpd["Region"],
    dropna=False
)

class_region_counts


In [ ]:
# Visualize class-region data coverage as a heatmap.
# This is a coverage heatmap, not a risk heatmap.
plt.figure(figsize=(10, 6))
plt.imshow(class_region_counts, aspect="auto")
plt.colorbar(label="Number of population records")
plt.xticks(
    range(len(class_region_counts.columns)),
    class_region_counts.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(class_region_counts.index)),
    class_region_counts.index
)
plt.xlabel("Region")
plt.ylabel("Class")
plt.title("Data coverage by vertebrate class and region")
plt.tight_layout()
plt.show()


## 9. Initial exploration summary

This initial exploration shows that the dataset is suitable for a population-level biodiversity trend analysis, but it also highlights important limitations:

- the main analytical unit is the monitored population record;
- population records are not evenly distributed across vertebrate classes;
- regional and ecosystem coverage is uneven;
- temporal coverage varies by year;
- spatial analysis depends on the availability of coordinates.

These observations guide the next notebook, where population-level trends are calculated only for populations with sufficient monitoring duration.


In [ ]:
# Store a compact summary of basic dataset properties.
# This can be reused in the README or project presentation.
initial_summary = pd.DataFrame({
    "Metric": [
        "Rows / population records",
        "Columns",
        "Unique population IDs",
        "Unique species names",
        "Unique binomial names",
        "First observation year column",
        "Last observation year column"
    ],
    "Value": [
        lpd.shape[0],
        lpd.shape[1],
        lpd["ID"].nunique(),
        lpd["Species"].nunique(),
        lpd["Binomial"].nunique(),
        min(years),
        max(years)
    ]
})

initial_summary
